# DubFlow — Google Colab TTS service

Run the four cells below. The last cell prints the URL and token to copy into the application's `.env` file. This notebook only serves multilingual MMS-TTS; transcription, translation, orchestration, and job storage stay in the main application.

In [ ]:
REPO_URL = "https://github.com/huynhphatloi/MultilingualVideoDubbingSystem.git"
BRANCH = "main"
PORT = 8000

In [ ]:
from pathlib import Path

if not Path("/content/dubflow").exists():
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/dubflow
%cd /content/dubflow/colab
!pip install -q -r requirements.txt

In [ ]:
import os
import secrets
import subprocess
import threading
import time

import uvicorn

AUTH_TOKEN = secrets.token_urlsafe(24)
os.environ["AUTH_TOKEN"] = AUTH_TOKEN

import server

api_thread = threading.Thread(
    target=lambda: uvicorn.run(
        server.app, host="0.0.0.0", port=PORT, log_level="warning"
    ),
    daemon=True,
)
api_thread.start()
time.sleep(2)
print("TTS API started on the Colab runtime.")

In [ ]:
import re

cloudflared = Path("/content/cloudflared")
if not cloudflared.exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

tunnel = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
public_url = None
for line in iter(tunnel.stdout.readline, ""):
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError("Cloudflare tunnel did not return a public URL")

print("Copy these values into the project .env file:")
print(f"COLAB_TTS_URL={public_url}")
print(f"COLAB_TTS_TOKEN={AUTH_TOKEN}")
print("COLAB_TTS_TIMEOUT=300")